In [1]:
# Load the data
import numpy as np
# Path to the saved .npy files
spectrograms_file = "dataSpectogram/spectrograms.npy"
targets_file = "dataSpectogram/targets.npy"

# Load the spectrograms and targets
spectrograms = np.load(spectrograms_file)
targets = np.load(targets_file)

# Verify the data
print(f"Spectrograms shape: {spectrograms.shape}")  # Should be (num_samples, 227, fixed_time_dim)
print(f"Targets shape: {targets.shape}")  # Should be (num_samples,)


Spectrograms shape: (30000, 227, 169)
Targets shape: (30000,)


In [2]:
# Shuffle the data
indices = np.arange(len(spectrograms))
np.random.shuffle(indices)

spectrograms = spectrograms[indices]
targets = targets[indices]


In [3]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Use only 60% of the data
num_samples = len(spectrograms)
num_training_samples = int(0.6 * num_samples)

# Split data into 60% usable data and 40% for adversarial attack
usable_spectrograms = spectrograms[:num_training_samples]
usable_targets = targets[:num_training_samples]
# Expand dimensions to match AlexNet input format (add a channel dimension)
usable_spectrograms = usable_spectrograms[..., np.newaxis]  # Shape: (num_samples, 227, 169, 1)

adversarial_spectrograms = spectrograms[num_training_samples:]
adversarial_targets = targets[num_training_samples:]

# Verify splits
print(f"Usable data: {len(usable_spectrograms)} samples")
print(f"Adversarial data: {len(adversarial_spectrograms)} samples")

# Further split the 60% usable data into training and testing sets
# Convert targets to one-hot encoding
num_classes = len(np.unique(targets))
targets_one_hot = to_categorical(usable_targets, num_classes=num_classes)

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(usable_spectrograms, targets_one_hot, test_size=0.2, random_state=42)

print(f"Training data: {len(X_train)} samples")
print(f"Testing data: {len(X_val)} samples")

2025-10-29 10:58:10.129378: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-29 10:58:10.173636: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-10-29 10:58:10.173667: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-10-29 10:58:10.173699: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-29 10:58:10.183695: I tensorflow/core/platform/cpu_feature_g

Usable data: 18000 samples
Adversarial data: 12000 samples
Training data: 14400 samples
Testing data: 3600 samples


In [4]:
import tensorflow as tf

# Load fixed model
model = tf.keras.models.load_model("alexnet_spectrogram_model.h5")

# Verify the model
model.summary()

2025-10-29 10:58:17.012636: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1886] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20854 MB memory:  -> device: 0, name: NVIDIA A10, pci bus id: 0000:17:00.0, compute capability: 8.6
2025-10-29 10:58:17.015199: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1886] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 18943 MB memory:  -> device: 1, name: NVIDIA A10, pci bus id: 0000:ca:00.0, compute capability: 8.6


Model: "AlexNet"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 data (InputLayer)           [(None, 227, 169, 1)]     0         
                                                                 
 conv1 (Conv2D)              (None, 55, 40, 96)        11712     
                                                                 
 relu1 (ReLU)                (None, 55, 40, 96)        0         
                                                                 
 pool1 (MaxPooling2D)        (None, 27, 19, 96)        0         
                                                                 
 conv2 (Conv2D)              (None, 27, 19, 256)       307456    
                                                                 
 relu2 (ReLU)                (None, 27, 19, 256)       0         
                                                                 
 pool2 (MaxPooling2D)        (None, 13, 9, 256)        0   

In [5]:
# Evaluate on validation data to see how well it predicts
val_loss, val_accuracy = model.evaluate(X_val, y_val, verbose=1)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

2025-10-12 15:43:20.228667: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:442] Loaded cuDNN version 8700
2025-10-12 15:43:20.559953: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x7f5fe5eb2ab0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-10-12 15:43:20.559974: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
2025-10-12 15:43:20.559978: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
2025-10-12 15:43:21.431574: I ./tensorflow/compiler/jit/device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


113/113 [==============================] - 5s 20ms/step - loss: 0.0646 - accuracy: 0.9811
Validation Loss: 0.0646
Validation Accuracy: 0.9811


In [5]:
import tensorflow as tf

def pgd_attack(model, spectrogram, label, epsilon=0.1, alpha=0.01, num_iter=40):

    # Clone the spectrogram to avoid modifying the original input
    perturbed_spectrogram = tf.cast(tf.identity(spectrogram), tf.float32)  # Ensure it's float32

    # Add random initialization within the epsilon ball
    perturbed_spectrogram += tf.random.uniform(perturbed_spectrogram.shape, minval=-epsilon, maxval=epsilon, dtype=tf.float32)
    perturbed_spectrogram = tf.clip_by_value(perturbed_spectrogram, clip_value_min=0.0, clip_value_max=1.0)

    # Perform iterative attack
    for _ in range(num_iter):
        with tf.GradientTape() as tape:
            tape.watch(perturbed_spectrogram)

            # Forward pass
            predictions = model(perturbed_spectrogram)

            # Compute loss (cross-entropy)
            loss = tf.keras.losses.categorical_crossentropy(label, predictions)

        # Compute gradients of loss with respect to input
        gradient = tape.gradient(loss, perturbed_spectrogram)
        signed_grad = tf.sign(gradient)

        # Update the perturbed spectrogram
        perturbed_spectrogram += alpha * signed_grad

        # Project back to the epsilon-ball and ensure valid pixel range
        perturbation = tf.clip_by_value(perturbed_spectrogram - spectrogram, -epsilon, epsilon)
        perturbed_spectrogram = tf.clip_by_value(spectrogram + perturbation, clip_value_min=0.0, clip_value_max=1.0)

    return perturbed_spectrogram


In [7]:
# Example spectrogram and label
index = 150  # Choose an index from the validation set
sample_spectrogram = X_val[index]  
label = y_val[index]  

# Convert to TensorFlow tensors
sample_spectrogram = tf.convert_to_tensor(sample_spectrogram, dtype=tf.float32)
label = tf.convert_to_tensor(label, dtype=tf.float32)  # Ensure label is float32

# Add batch and channel dimensions to spectrogram
sample_spectrogram = tf.expand_dims(sample_spectrogram, axis=0)  # Add batch dimension
sample_spectrogram = tf.expand_dims(sample_spectrogram, axis=-1)  # Add channel dimension
label = tf.expand_dims(label, axis=0)  # Add batch dimension to match predictions


# Perform PGD attack
adversarial_example = pgd_attack(model, sample_spectrogram, label, epsilon=0.1, alpha=0.01, num_iter=40)

# Get predictions for the adversarial example
predictions = model(adversarial_example)
predicted_class = tf.argmax(predictions, axis=1)

print("Predicted class for adversarial example:", predicted_class.numpy()[0])
print('True class:', np.argmax(label))


Predicted class for adversarial example: 7
True class: 1


In [12]:
# Check how well the model is doing on validation data with adversarial attack
def evaluate_pgd_attack(model, X_val, y_val, epsilon=0.1, alpha=0.01, 
                                num_iter=40, batch_size=32, max_samples=1000):
    """
    Improved batched version - processes multiple samples at once
    Much faster than sample-by-sample evaluation
    """
    total_samples = min(len(X_val), max_samples)
    correct_predictions = 0
    samples_processed = 0
    
    # Process in batches
    num_batches = (total_samples + batch_size - 1) // batch_size
    
    for batch_idx in range(num_batches):
        start_idx = batch_idx * batch_size
        end_idx = min(start_idx + batch_size, total_samples)
        
        # Get batch
        X_batch = tf.convert_to_tensor(X_val[start_idx:end_idx], dtype=tf.float32)
        y_batch = tf.convert_to_tensor(y_val[start_idx:end_idx], dtype=tf.float32)
        
        # Generate adversarial examples for entire batch
        X_adv_batch = pgd_attack_batch(model, X_batch, y_batch, epsilon, alpha, num_iter)
        
        # Evaluate on adversarial batch
        predictions = model(X_adv_batch)
        predicted_classes = tf.argmax(predictions, axis=1).numpy()
        true_classes = tf.argmax(y_batch, axis=1).numpy()
        
        # Count correct predictions
        correct_predictions += np.sum(predicted_classes == true_classes)
        samples_processed = end_idx
        
        # Progress
        if (batch_idx + 1) % max(1, num_batches // 10) == 0 or batch_idx == num_batches - 1:
            percent_done = int(samples_processed / total_samples * 100)
            print(f"Progress: {percent_done}% ({samples_processed}/{total_samples})")
    
    adversarial_accuracy = correct_predictions / total_samples
    return adversarial_accuracy


In [6]:
def pgd_attack_batch(model, x_batch, y_batch, epsilon=0.1, alpha=0.01, num_iter=40):
    """
    PGD attack that works on batches
    """
    # Clone input
    x_adv = tf.identity(x_batch)
    
    # Random initialization
    x_adv = x_adv + tf.random.uniform(tf.shape(x_adv), minval=-epsilon, maxval=epsilon)
    x_adv = tf.clip_by_value(x_adv, 0.0, 1.0)
    
    # PGD iterations
    for _ in range(num_iter):
        with tf.GradientTape() as tape:
            tape.watch(x_adv)
            predictions = model(x_adv)
            loss = tf.keras.losses.categorical_crossentropy(y_batch, predictions)
        
        # Compute gradient
        gradient = tape.gradient(loss, x_adv)
        signed_grad = tf.sign(gradient)
        
        # Update
        x_adv = x_adv + alpha * signed_grad
        
        # Project back
        perturbation = tf.clip_by_value(x_adv - x_batch, -epsilon, epsilon)
        x_adv = tf.clip_by_value(x_batch + perturbation, 0.0, 1.0)
    
    return x_adv

In [12]:
# Evaluate model under PGD attack
adversarial_accuracy = evaluate_pgd_attack(model, X_val, y_val, epsilon=0.1, alpha=0.01, num_iter=40)
print(f"Accuracy of normal model under PGD Attack: {adversarial_accuracy * 100:.2f}%")

Progress: 10% (100/1000)
Progress: 20% (200/1000)
Progress: 30% (300/1000)
Progress: 40% (400/1000)
Progress: 50% (500/1000)
Progress: 60% (600/1000)
Progress: 70% (700/1000)
Progress: 80% (800/1000)
Progress: 90% (900/1000)
Progress: 100% (1000/1000)
Accuracy of normal model under PGD Attack: 10.70%


In [8]:
# Defense distillation method
# Get logits before the final softmax layer
teacher_logits_model = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer('fc8').output  # pre-softmax logits
)

train_logits = teacher_logits_model.predict(X_train, batch_size=32)
val_logits = teacher_logits_model.predict(X_val, batch_size=32)

# Apply softmax with temperature
temperature = 10
y_train_soft = tf.nn.softmax(train_logits / temperature).numpy()
y_val_soft = tf.nn.softmax(val_logits / temperature).numpy()

print(f"Train soft labels shape: {y_train_soft.shape}, val soft labels shape: {y_val_soft.shape}")

# Check the soft targets
np.set_printoptions(precision=4, suppress=True)  # 4 decimals, no scientific notation
print("Sample soft labels:")
print(y_train_soft[:3])  # first 3 samples

113/113 [==============================] - 1s 7ms/step
Train soft labels shape: (14400, 10), val soft labels shape: (3600, 10)
Sample soft labels:
[[0.055  0.0532 0.0483 0.0696 0.0441 0.0529 0.0479 0.5177 0.0466 0.0647]
 [0.0225 0.0213 0.0203 0.0202 0.0286 0.0172 0.0128 0.8236 0.011  0.0225]
 [0.0027 0.0028 0.0032 0.0028 0.0049 0.0022 0.0016 0.9752 0.001  0.0035]]


In [9]:
# Cell 10: Create and train student model with distillation
# Create student model (same architecture as teacher)
student_model = tf.keras.models.clone_model(model)
student_model.set_weights(model.get_weights())  # Initialize with teacher weights

print("Student model created with same architecture as teacher")

# Create a custom training class for distillation
class DistillationModel(tf.keras.Model):
    def __init__(self, base_model, temperature=10):
        super().__init__()
        self.base_model = base_model
        self.temperature = temperature
        
    def call(self, inputs, training=None):
        return self.base_model(inputs, training=training)
    
    @tf.function
    def train_step(self, data):
        x, y_soft = data
        
        with tf.GradientTape() as tape:
            # Get the base model predictions (this will be the full forward pass)
            predictions = self.base_model(x, training=True)
            
            # Get logits from the fc8 layer
            # We need to recreate the forward pass up to fc8
            conv_layers = []
            for layer in self.base_model.layers:
                if 'conv' in layer.name or 'pool' in layer.name or 'relu' in layer.name or 'flatten' in layer.name or 'fc' in layer.name:
                    conv_layers.append(layer)
                if layer.name == 'fc8':
                    break
            
            # Forward pass to get logits
            logits = x
            for layer in conv_layers:
                logits = layer(logits, training=True)
            
            # Apply temperature scaling
            soft_preds = tf.nn.softmax(logits / self.temperature)
            
            # KL divergence loss (scaled by temperature^2)
            loss = tf.keras.losses.KLDivergence()(y_soft, soft_preds) * (self.temperature ** 2)
        
        # Update weights
        gradients = tape.gradient(loss, self.base_model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.base_model.trainable_variables))
        
        return {"loss": loss}
    
    @tf.function
    def test_step(self, data):
        x, y_soft = data
        
        # Get logits from fc8 layer
        conv_layers = []
        for layer in self.base_model.layers:
            if 'conv' in layer.name or 'pool' in layer.name or 'relu' in layer.name or 'flatten' in layer.name or 'fc' in layer.name:
                conv_layers.append(layer)
            if layer.name == 'fc8':
                break
        
        # Forward pass to get logits
        logits = x
        for layer in conv_layers:
            logits = layer(logits, training=False)
        
        # Apply temperature scaling
        soft_preds = tf.nn.softmax(logits / self.temperature)
        
        # KL divergence loss
        loss = tf.keras.losses.KLDivergence()(y_soft, soft_preds) * (self.temperature ** 2)
        
        return {"loss": loss}

# Create distillation model
distill_model = DistillationModel(student_model, temperature=temperature)
distill_model.compile(optimizer='adam')

print("Starting distillation training...")

# Train the student model with distillation
history = distill_model.fit(
    X_train, y_train_soft,
    validation_data=(X_val, y_val_soft),
    epochs=30,
    batch_size=32,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)],
    verbose=1
)

Student model created with same architecture as teacher
Starting distillation training...
Epoch 1/30


2025-10-12 15:43:39.091150: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


450/450 [==============================] - 20s 30ms/step - loss: 1.5037 - val_loss: 2.8898
Epoch 2/30
450/450 [==============================] - 11s 25ms/step - loss: 1.2386 - val_loss: 1.1524
Epoch 3/30
450/450 [==============================] - 11s 26ms/step - loss: 1.0697 - val_loss: 0.7411
Epoch 4/30
450/450 [==============================] - 12s 27ms/step - loss: 0.9131 - val_loss: 1.8741
Epoch 5/30
450/450 [==============================] - 13s 28ms/step - loss: 1.0149 - val_loss: 1.6480
Epoch 6/30
450/450 [==============================] - 12s 27ms/step - loss: 0.9600 - val_loss: 1.8241
Epoch 7/30
450/450 [==============================] - 12s 27ms/step - loss: 1.0138 - val_loss: 1.3500
Epoch 8/30
450/450 [==============================] - 12s 27ms/step - loss: 0.8996 - val_loss: 1.4068


In [15]:
# Cell 11: Evaluate student model on clean data
# Compile the student model for standard evaluation
student_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Evaluate on clean data
student_val_loss, student_val_accuracy = student_model.evaluate(X_val, y_val, verbose=1)
print(f"Student Model - Validation Loss: {student_val_loss:.4f}")
print(f"Student Model - Validation Accuracy: {student_val_accuracy:.4f}")

113/113 [==============================] - 2s 12ms/step - loss: 0.0970 - accuracy: 0.9758
Student Model - Validation Loss: 0.0970
Student Model - Validation Accuracy: 0.9758


In [16]:
# Evaluate student model under PGD attack
student_adversarial_accuracy = evaluate_pgd_attack(student_model, X_val, y_val, epsilon=0.1, alpha=0.01, num_iter=40)
print(f"Student Model Accuracy under PGD Attack: {student_adversarial_accuracy * 100:.2f}%")

Progress: 10% (100/1000)
Progress: 20% (200/1000)
Progress: 30% (300/1000)
Progress: 40% (400/1000)
Progress: 50% (500/1000)
Progress: 60% (600/1000)
Progress: 70% (700/1000)
Progress: 80% (800/1000)
Progress: 90% (900/1000)
Progress: 100% (1000/1000)
Student Model Accuracy under PGD Attack: 10.80%


In [10]:
# Try the robust training method only and check the performance
# Cell 1: Create Adversarial Training Model Class
class AdversarialTrainingModel(tf.keras.Model):
    """
    Model that trains on adversarially perturbed examples
    """
    def __init__(self, base_model, epsilon=0.1, alpha=0.01, num_iter=10):
        super().__init__()
        self.base_model = base_model
        self.epsilon = epsilon
        self.alpha = alpha
        self.num_iter = num_iter
        
    def call(self, inputs, training=None):
        return self.base_model(inputs, training=training)
    
    def pgd_attack_batch(self, x, y_true):
        """
        Generate adversarial examples for a batch using PGD
        """
        # Clone the input
        x_adv = tf.identity(x)
        
        # Random initialization within epsilon ball
        x_adv = x_adv + tf.random.uniform(tf.shape(x_adv), minval=-self.epsilon, maxval=self.epsilon)
        x_adv = tf.clip_by_value(x_adv, 0.0, 1.0)
        
        # Iterative PGD
        for _ in range(self.num_iter):
            with tf.GradientTape() as tape:
                tape.watch(x_adv)
                predictions = self.base_model(x_adv, training=True)
                loss = tf.keras.losses.categorical_crossentropy(y_true, predictions)
            
            # Compute gradient
            gradient = tape.gradient(loss, x_adv)
            signed_grad = tf.sign(gradient)
            
            # Update adversarial example
            x_adv = x_adv + self.alpha * signed_grad
            
            # Project back to epsilon ball
            perturbation = tf.clip_by_value(x_adv - x, -self.epsilon, self.epsilon)
            x_adv = tf.clip_by_value(x + perturbation, 0.0, 1.0)
        
        return x_adv
    
    @tf.function
    def train_step(self, data):
        x, y = data
        
        # Generate adversarial examples for the entire batch
        x_adv = self.pgd_attack_batch(x, y)
        
        # Train on adversarial examples
        with tf.GradientTape() as tape:
            predictions = self.base_model(x_adv, training=True)
            loss = self.compiled_loss(y, predictions)
        
        # Update weights
        gradients = tape.gradient(loss, self.base_model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.base_model.trainable_variables))
        
        # Update metrics
        self.compiled_metrics.update_state(y, predictions)
        
        return {m.name: m.result() for m in self.metrics}
    
    @tf.function
    def test_step(self, data):
        x, y = data
        
        # Generate adversarial examples for validation
        x_adv = self.pgd_attack_batch(x, y)
        
        # Evaluate on adversarial examples
        predictions = self.base_model(x_adv, training=False)
        loss = self.compiled_loss(y, predictions)
        
        # Update metrics
        self.compiled_metrics.update_state(y, predictions)
        
        return {m.name: m.result() for m in self.metrics}

In [11]:
# Cell 2: Create Student Model for Adversarial Training
print("="*60)
print("ADVERSARIAL TRAINING - TRAINING ON ADVERSARIAL EXAMPLES ONLY")
print("="*60)

# Create a fresh student model (clone from teacher)
student_adv = tf.keras.models.clone_model(model)
student_adv.set_weights(model.get_weights())

print("\nStudent model created (initialized with teacher weights)")
print(f"Training data: {len(X_train)} samples")
print(f"Validation data: {len(X_val)} samples")

ADVERSARIAL TRAINING - TRAINING ON ADVERSARIAL EXAMPLES ONLY

Student model created (initialized with teacher weights)
Training data: 14400 samples
Validation data: 3600 samples


In [12]:
# Cell 3: Configure Adversarial Training
# Hyperparameters for adversarial training
epsilon = 0.1      # Perturbation budget (same as attack during testing)
alpha = 0.01       # Step size for PGD
num_iter = 10      # Number of PGD iterations (reduced from 40 for speed)

print("\nAdversarial Training Configuration:")
print(f"  Epsilon (perturbation budget): {epsilon}")
print(f"  Alpha (step size): {alpha}")
print(f"  PGD iterations per batch: {num_iter}")
print(f"  Note: Training will be slower due to adversarial generation")


Adversarial Training Configuration:
  Epsilon (perturbation budget): 0.1
  Alpha (step size): 0.01
  PGD iterations per batch: 10
  Note: Training will be slower due to adversarial generation


In [13]:
# Cell 4: Create and Compile Adversarial Training Model
adv_training_model = AdversarialTrainingModel(
    base_model=student_adv,
    epsilon=epsilon,
    alpha=alpha,
    num_iter=num_iter
)

adv_training_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),  # Lower learning rate for stability
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nAdversarial training model compiled")
print("Ready to train on adversarial examples!")


Adversarial training model compiled
Ready to train on adversarial examples!


In [26]:
# Cell 5: Train with Adversarial Examples
print("\n" + "="*60)
print("STARTING ADVERSARIAL TRAINING")
print("="*60)
print("This will take longer than normal training...")
print("Each batch generates adversarial examples on-the-fly\n")

# Train the model
history_adv = adv_training_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=7, 
            restore_best_weights=True,
            monitor='val_loss'
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-6
        )
    ],
    verbose=1
)

print("\nAdversarial training completed!")


STARTING ADVERSARIAL TRAINING
This will take longer than normal training...
Each batch generates adversarial examples on-the-fly

Epoch 1/30
450/450 [==============================] - 98s 218ms/step - loss: 2.1237 - accuracy: 0.1784 - val_loss: 2.1206 - val_accuracy: 0.1750 - lr: 5.0000e-05
Epoch 2/30
450/450 [==============================] - 100s 222ms/step - loss: 2.1255 - accuracy: 0.1775 - val_loss: 2.1197 - val_accuracy: 0.1764 - lr: 5.0000e-05
Epoch 3/30
450/450 [==============================] - 100s 223ms/step - loss: 2.1251 - accuracy: 0.1804 - val_loss: 2.1196 - val_accuracy: 0.1783 - lr: 5.0000e-05
Epoch 4/30
450/450 [==============================] - 101s 224ms/step - loss: 2.1231 - accuracy: 0.1749 - val_loss: 2.1192 - val_accuracy: 0.1761 - lr: 5.0000e-05
Epoch 5/30
450/450 [==============================] - 101s 224ms/step - loss: 2.1221 - accuracy: 0.1763 - val_loss: 2.1189 - val_accuracy: 0.1836 - lr: 5.0000e-05
Epoch 6/30
450/450 [==============================] - 1

In [15]:
# Cell 6: Evaluate Student Model on Clean Data
print("\n" + "="*60)
print("EVALUATION ON CLEAN DATA")
print("="*60)

# Compile the base model for normal evaluation
student_adv.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Evaluate on clean validation data
clean_loss, clean_accuracy = student_adv.evaluate(X_val, y_val, verbose=1)
print(f"Student Advesarial Model - Validation Loss: {clean_loss:.4f}")
print(f"Student Advesarial Model - Validation Accuracy: {clean_accuracy:.4f}")


EVALUATION ON CLEAN DATA
113/113 [==============================] - 1s 11ms/step - loss: 13.0007 - accuracy: 0.1075
Student Advesarial Model - Validation Loss: 13.0007
Student Advesarial Model - Validation Accuracy: 0.1075


In [50]:
# Evaluate student model (advesarial) under PGD attack
student_adversarial_accuracy = evaluate_pgd_attack(adv_training_model, X_val, y_val, epsilon=0.1, alpha=0.01, num_iter=10)
print(f"Student Model Accuracy under PGD Attack: {student_adversarial_accuracy * 100:.2f}%")

Progress: 9% (96/1000)
Progress: 19% (192/1000)
Progress: 28% (288/1000)
Progress: 38% (384/1000)
Progress: 48% (480/1000)
Progress: 57% (576/1000)
Progress: 67% (672/1000)
Progress: 76% (768/1000)
Progress: 86% (864/1000)
Progress: 96% (960/1000)
Progress: 100% (1000/1000)
Student Model Accuracy under PGD Attack: 17.40%


In [19]:
# ============================================================
# MIXED ADVERSARIAL TRAINING - Clean + Adversarial Examples
# ============================================================

import tensorflow as tf
import numpy as np

print("="*70)
print("MIXED ADVERSARIAL TRAINING - TRAINING ON BOTH CLEAN AND ADVERSARIAL")
print("="*70)
print("\nThis approach trains on a mix of:")
print("  - Clean examples (to maintain clean accuracy)")
print("  - Adversarial examples (to build robustness)")
print("  Result: Better balance between clean and adversarial performance\n")

# ============================================================
# Step 1: Define Mixed Adversarial Training Model Class
# ============================================================

class MixedAdversarialTrainingModel(tf.keras.Model):
    """
    Optimized model with batched adversarial example generation
    Much faster than sample-by-sample processing
    """
    def __init__(self, base_model, epsilon=0.1, alpha=0.01, num_iter=20, mix_ratio=0.5):
        super().__init__()
        self.base_model = base_model
        self.epsilon = epsilon
        self.alpha = alpha
        self.num_iter = num_iter
        self.mix_ratio = mix_ratio
        
    def call(self, inputs, training=None):
        return self.base_model(inputs, training=training)
    
    @tf.function
    def pgd_attack_batch_optimized(self, x_batch, y_batch):
        """
        OPTIMIZED: Vectorized PGD attack for entire batch
        Processes all samples simultaneously for better GPU utilization
        """
        batch_size = tf.shape(x_batch)[0]
        
        # Initialize perturbation with random noise
        delta = tf.random.uniform(
            tf.shape(x_batch), 
            minval=-self.epsilon, 
            maxval=self.epsilon,
            dtype=tf.float32
        )
        
        # Ensure valid range after initialization
        x_adv = tf.clip_by_value(x_batch + delta, 0.0, 1.0)
        
        # PGD iterations - vectorized over entire batch
        for _ in tf.range(self.num_iter):
            with tf.GradientTape() as tape:
                tape.watch(x_adv)
                # Forward pass for entire batch
                predictions = self.base_model(x_adv, training=False)
                # Compute loss for all samples at once
                loss = tf.reduce_sum(
                    tf.keras.losses.categorical_crossentropy(y_batch, predictions)
                )
            
            # Compute gradients for entire batch
            gradients = tape.gradient(loss, x_adv)
            
            # Take sign and update (vectorized)
            signed_grads = tf.sign(gradients)
            x_adv = x_adv + self.alpha * signed_grads
            
            # Project back to epsilon ball and valid range (vectorized)
            delta = tf.clip_by_value(x_adv - x_batch, -self.epsilon, self.epsilon)
            x_adv = tf.clip_by_value(x_batch + delta, 0.0, 1.0)
        
        return x_adv
    
    @tf.function
    def train_step(self, data):
        x, y = data
        batch_size = tf.shape(x)[0]
        
        # Calculate split point
        num_adversarial = tf.cast(
            tf.cast(batch_size, tf.float32) * self.mix_ratio, 
            tf.int32
        )
        num_clean = batch_size - num_adversarial
        
        # Split batch
        x_clean = x[:num_clean]
        y_clean = y[:num_clean]
        x_to_attack = x[num_clean:]
        y_to_attack = y[num_clean:]
        
        # Generate adversarial examples for entire adversarial portion at once
        # This is the key optimization - batched processing
        x_adv = self.pgd_attack_batch_optimized(x_to_attack, y_to_attack)
        
        # Combine clean and adversarial
        x_combined = tf.concat([x_clean, x_adv], axis=0)
        y_combined = tf.concat([y_clean, y_to_attack], axis=0)
        
        # Training step
        with tf.GradientTape() as tape:
            predictions = self.base_model(x_combined, training=True)
            loss = self.compiled_loss(y_combined, predictions)
        
        # Update weights
        gradients = tape.gradient(loss, self.base_model.trainable_variables)
        self.optimizer.apply_gradients(
            zip(gradients, self.base_model.trainable_variables)
        )
        
        # Update metrics
        self.compiled_metrics.update_state(y_combined, predictions)
        
        return {m.name: m.result() for m in self.metrics}
    
    @tf.function
    def test_step(self, data):
        """Validate on clean data"""
        x, y = data
        predictions = self.base_model(x, training=False)
        loss = self.compiled_loss(y, predictions)
        self.compiled_metrics.update_state(y, predictions)
        return {m.name: m.result() for m in self.metrics}

# ============================================================
# Step 2: Create Fresh Student Model for Mixed Training
# ============================================================

print("Creating new student model for mixed adversarial training...")
student_mixed = tf.keras.models.clone_model(model)
student_mixed.set_weights(model.get_weights())

print(f"✓ Student model created (initialized with teacher weights)")
print(f"✓ Training data: {len(X_train)} samples")
print(f"✓ Validation data: {len(X_val)} samples")


# ============================================================
# Step 3: Configure Mixed Adversarial Training
# ============================================================

epsilon = 0.1      # Perturbation budget
alpha = 0.1       # Step size for PGD
num_iter = 20      # PGD iterations (reduced for speed during training)
mix_ratio = 0.9    # 50% clean, 50% adversarial

print("\n" + "="*70)
print("CONFIGURATION")
print("="*70)
print(f"Epsilon (perturbation budget):  {epsilon}")
print(f"Alpha (step size):              {alpha}")
print(f"PGD iterations per batch:       {num_iter}")
print(f"Mix ratio:                      {int(mix_ratio*100)}% adversarial, {int((1-mix_ratio)*100)}% clean")
print("="*70)

MIXED ADVERSARIAL TRAINING - TRAINING ON BOTH CLEAN AND ADVERSARIAL

This approach trains on a mix of:
  - Clean examples (to maintain clean accuracy)
  - Adversarial examples (to build robustness)
  Result: Better balance between clean and adversarial performance

Creating new student model for mixed adversarial training...
✓ Student model created (initialized with teacher weights)
✓ Training data: 14400 samples
✓ Validation data: 3600 samples

CONFIGURATION
Epsilon (perturbation budget):  0.1
Alpha (step size):              0.1
PGD iterations per batch:       20
Mix ratio:                      90% adversarial, 9% clean


In [20]:
# ============================================================
# Step 4: Create and Compile Mixed Training Model
# ============================================================

mixed_training_model = MixedAdversarialTrainingModel(
    base_model=student_mixed,
    epsilon=epsilon,
    alpha=alpha,
    num_iter=num_iter,
    mix_ratio=mix_ratio
)

mixed_training_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n✓ Mixed adversarial training model compiled and ready!")


✓ Mixed adversarial training model compiled and ready!


In [21]:
# ============================================================
# Step 5: Train with Mixed Examples
# ============================================================

print("\n" + "="*70)
print("STARTING MIXED ADVERSARIAL TRAINING")
print("="*70)
print("This will take longer than normal training but faster than pure adversarial...")
print("Estimated time: ~2-3 minutes per epoch\n")

history_mixed = mixed_training_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=128,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            patience=7, 
            restore_best_weights=True,
            monitor='val_accuracy',
            verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=7,
            min_lr=1e-6,
            verbose=1
        )
    ],
    verbose=1
)

print("\n✓ Mixed adversarial training completed!")



STARTING MIXED ADVERSARIAL TRAINING
This will take longer than normal training but faster than pure adversarial...
Estimated time: ~2-3 minutes per epoch

Epoch 1/30


2025-10-29 11:32:28.151873: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] layout failed: INVALID_ARGUMENT: MutableGraphView::SortTopologically error: detected edge(s) creating cycle(s) {'StatefulPartitionedCall/StatefulPartitionedCall/while/next_iteration/_135-0-0-TransposeNCHWToNHWC-LayoutOptimizer' -> 'StatefulPartitionedCall/StatefulPartitionedCall/while/merge/_108'}.


113/113 [==============================] - 56s 449ms/step - loss: 2.0827 - accuracy: 0.1922 - val_loss: 0.0336 - val_accuracy: 0.9914 - lr: 1.0000e-04
Epoch 2/30
113/113 [==============================] - 48s 427ms/step - loss: 2.0348 - accuracy: 0.2188 - val_loss: 0.0388 - val_accuracy: 0.9906 - lr: 1.0000e-04
Epoch 3/30
113/113 [==============================] - 48s 424ms/step - loss: 1.9648 - accuracy: 0.2417 - val_loss: 0.0250 - val_accuracy: 0.9950 - lr: 1.0000e-04
Epoch 4/30
113/113 [==============================] - 49s 432ms/step - loss: 1.9367 - accuracy: 0.2478 - val_loss: 0.0389 - val_accuracy: 0.9922 - lr: 1.0000e-04
Epoch 5/30
113/113 [==============================] - 48s 425ms/step - loss: 1.9243 - accuracy: 0.2585 - val_loss: 0.0390 - val_accuracy: 0.9908 - lr: 1.0000e-04
Epoch 6/30
113/113 [==============================] - 49s 432ms/step - loss: 1.9219 - accuracy: 0.2565 - val_loss: 0.0354 - val_accuracy: 0.9917 - lr: 1.0000e-04
Epoch 7/30
113/113 [===================

In [22]:
# ============================================================
# Step 6: Evaluate on Clean Data
# ============================================================

print("\n" + "="*70)
print("EVALUATION ON CLEAN DATA")
print("="*70)

# Compile for evaluation
student_mixed.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Evaluate on clean validation data
clean_loss_mixed, clean_accuracy_mixed = student_mixed.evaluate(X_val, y_val, verbose=1)
print(f"\nMixed Adversarial Model - Clean Data Performance:")
print(f"  Loss:     {clean_loss_mixed:.4f}")
print(f"  Accuracy: {clean_accuracy_mixed:.4f} ({clean_accuracy_mixed * 100:.2f}%)")



EVALUATION ON CLEAN DATA
113/113 [==============================] - 2s 9ms/step - loss: 0.0250 - accuracy: 0.9950

Mixed Adversarial Model - Clean Data Performance:
  Loss:     0.0250
  Accuracy: 0.9950 (99.50%)


In [24]:
# ============================================================
# Step 7: Evaluate on Adversarial Examples
# ============================================================
adv_iter = 10
print("\n" + "="*70)
print("EVALUATION ON ADVERSARIAL EXAMPLES")
print("="*70)
print(f"Testing with strong PGD attack (epsilon=0.1, iterations={adv_iter})")

# Evaluate under PGD attack
mixed_adversarial_accuracy = evaluate_pgd_attack(
    student_mixed, X_val, y_val, 
    epsilon=0.1, 
    alpha=0.1, 
    num_iter=adv_iter
)

print(f"\nMixed Adversarial Model - Adversarial Performance:")
print(f"  Accuracy under PGD attack: {mixed_adversarial_accuracy:.4f} ({mixed_adversarial_accuracy * 100:.2f}%)")



EVALUATION ON ADVERSARIAL EXAMPLES
Testing with strong PGD attack (epsilon=0.1, iterations=10)
Progress: 9% (96/1000)
Progress: 19% (192/1000)
Progress: 28% (288/1000)
Progress: 38% (384/1000)
Progress: 48% (480/1000)
Progress: 57% (576/1000)
Progress: 67% (672/1000)
Progress: 76% (768/1000)
Progress: 86% (864/1000)
Progress: 96% (960/1000)
Progress: 100% (1000/1000)

Mixed Adversarial Model - Adversarial Performance:
  Accuracy under PGD attack: 0.1660 (16.60%)
